# LOD2 Dataset Exploratory Data Analysis (EDA)

This notebook loads the CityJSON LOD2 building graph data from `data/The Hague/LOD2` and explores the distribution of building graph sizes (number of vertices/nodes and edges).

In [2]:
import sys
import os
import numpy as np
import pandas as pd
import torch
import plotly.express as px
import plotly.io as pio
import matplotlib.pyplot as plt

# Set Plotly renderer for standard notebooks
try:
    plt.style.use('seaborn-v0_8-whitegrid')
except:
    plt.style.use('default')

# Add the project root to the system path so we can import the src package
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

print(f"Project root added to sys.path: {project_root}")

Project root added to sys.path: c:\Users\maxim\Projects\lod-generation


In [3]:
from src.dataset.dataset import CityJSONDataset

# Load LOD2 data
# normalize_coords is set to True as is standard for our experiments
dataset_dir = "../data/The Hague"
print(f"Loading LOD2 dataset from: {dataset_dir}")
dataset = CityJSONDataset(dataset_dir=dataset_dir, lods=2, normalize_coords=True)

print(f"\nDataset successfully loaded!")
print(f"Total buildings in dataset: {len(dataset)}")
print(f"Resolved padding size (N_max): {dataset.n_max}")

c:\Users\maxim\anaconda3\envs\the_one_env\Lib\site-packages\torchvision\io\image.py:13: UserWarning:

Failed to load image Python extension: '[WinError 127] The specified procedure could not be found'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?



Loading LOD2 dataset from: ../data/The Hague


KeyboardInterrupt: 

In [ ]:
# Extract size metrics from raw graphs
stats = []

# lod_data[2] maps obj_id -> raw graph dict before padding
lod2_data = dataset.lod_data[2]

for obj_id in dataset.ids:
    graph = lod2_data[obj_id]
    num_nodes = graph["x"].shape[0]
    # In CityJSONDataset, edges are bidirectional (directed edges in edge_index)
    # We divide by 2 to get undirected edges count
    num_edges_dir = graph["edge_index"].shape[1]
    num_edges_undir = num_edges_dir // 2
    
    stats.append({
        "id": obj_id,
        "type": graph.get("type", "Unknown"),
        "nodes": num_nodes,
        "directed_edges": num_edges_dir,
        "undirected_edges": num_edges_undir,
    })

df = pd.DataFrame(stats)
print(f"Preview of building graph sizes (first 10 records):")
display(df.head(10))

## Summary Statistics of Graph Sizes

In [ ]:
# Compute summary statistics
summary_df = df[["nodes", "undirected_edges", "directed_edges"]].describe().T
# Add median (50%) to make it clearer
display(summary_df)

## Distribution Plots

In [ ]:
# Plot distribution of node counts
fig_nodes = px.histogram(
    df, 
    x="nodes", 
    nbins=40, 
    title="Distribution of Node Counts per Building Graph (LOD2)",
    labels={"nodes": "Number of Nodes (Vertices)"},
    color_discrete_sequence=["#636EFA"],
    marginal="box"  # Add boxplot on top
)
fig_nodes.update_layout(
    xaxis_title="Number of Nodes (Vertices)",
    yaxis_title="Count of Buildings",
    bargap=0.05,
    title_x=0.5
)
fig_nodes.show()

In [ ]:
# Plot distribution of undirected edge counts
fig_edges = px.histogram(
    df, 
    x="undirected_edges", 
    nbins=40, 
    title="Distribution of Undirected Edge Counts per Building Graph (LOD2)",
    labels={"undirected_edges": "Number of Undirected Edges"},
    color_discrete_sequence=["#EF553B"],
    marginal="box"  # Add boxplot on top
)
fig_edges.update_layout(
    xaxis_title="Number of Undirected Edges",
    yaxis_title="Count of Buildings",
    bargap=0.05,
    title_x=0.5
)
fig_edges.show()

## Relationship Between Nodes and Edges

In [ ]:
# Scatter plot comparing nodes vs edges
fig_scatter = px.scatter(
    df, 
    x="nodes", 
    y="undirected_edges", 
    color="type",
    hover_data=["id"],
    title="Relationship between Node and Undirected Edge Counts",
    labels={"nodes": "Number of Nodes (Vertices)", "undirected_edges": "Number of Undirected Edges"}
)
fig_scatter.update_layout(
    xaxis_title="Number of Nodes (Vertices)",
    yaxis_title="Number of Undirected Edges",
    title_x=0.5
)
fig_scatter.show()

## Size Distribution by Building Type

In [ ]:
# Distribution of building types
type_counts = df["type"].value_counts()
print("Building types frequency in the dataset:")
print(type_counts)

# Box plot to compare node size distribution by building type
fig_box = px.box(
    df, 
    x="type", 
    y="nodes", 
    color="type",
    title="Distribution of Nodes by Building Type",
    labels={"type": "Building Type", "nodes": "Number of Nodes"}
)
fig_box.update_layout(
    xaxis_title="Building Type",
    yaxis_title="Number of Nodes (Vertices)",
    title_x=0.5
)
fig_box.show()

## Visualization of Largest Graphs

We define a function `visualize_top_k_graphs` to visualize the top $k$ building graphs with the most nodes in 3D. This uses `plotly.graph_objects` to draw the vertices (nodes) and edges in 3D space, showing the structure of the building graphs.

In [ ]:
import plotly.graph_objects as go
import numpy as np
import torch

def visualize_top_k_graphs(dataset, k=5, lod=2):
    """
    Visualizes the top k graphs with the most nodes in the dataset.
    Each graph is rendered in 3D with a title containing the object ID and node count.
    
    Args:
        dataset (CityJSONDataset): The loaded dataset containing the graphs.
        k (int): Number of graphs to visualize.
        lod (int): Level of Detail (LOD) dataset to query (default is 2).
    """
    # Get graph dictionary for the selected LOD
    lod_data = dataset.lod_data[lod]
    
    # Sort building IDs by their graph's number of nodes in descending order
    sorted_ids = sorted(
        dataset.ids, 
        key=lambda obj_id: lod_data[obj_id]["x"].shape[0], 
        reverse=True
    )
    
    # Take the top k building IDs
    top_k_ids = sorted_ids[:k]
    
    for obj_id in top_k_ids:
        graph = lod_data[obj_id]
        
        # Extract coordinates (nodes) and convert to numpy if torch.Tensor
        nodes = graph["x"]
        if isinstance(nodes, torch.Tensor):
            nodes = nodes.numpy()
            
        # Extract edge list (edge_index) and convert to numpy if torch.Tensor
        edge_index = graph["edge_index"]
        if isinstance(edge_index, torch.Tensor):
            edge_index = edge_index.numpy()
            
        num_nodes = nodes.shape[0]
        
        # Construct lists of coordinates for the edges trace
        edge_x = []
        edge_y = []
        edge_z = []
        
        for u, v in zip(edge_index[0], edge_index[1]):
            if u < v:  # Only draw each undirected edge once
                edge_x.extend([nodes[u, 0], nodes[v, 0], None])
                edge_y.extend([nodes[u, 1], nodes[v, 1], None])
                edge_z.extend([nodes[u, 2], nodes[v, 2], None])
                
        fig = go.Figure()
        
        # Plot edges as 3D lines
        fig.add_trace(go.Scatter3d(
            x=edge_x,
            y=edge_y,
            z=edge_z,
            mode='lines',
            line=dict(color='#2c3e50', width=3),
            name='Edges'
        ))
        
        # Plot nodes as 3D markers
        fig.add_trace(go.Scatter3d(
            x=nodes[:, 0],
            y=nodes[:, 1],
            z=nodes[:, 2],
            mode='markers',
            marker=dict(size=4, color='#e74c3c'),
            name='Nodes'
        ))
        
        # Title includes the building ID and number of nodes
        title = f"Building: {obj_id} (Nodes: {num_nodes})"
        
        fig.update_layout(
            title=title,
            scene=dict(aspectmode='data'),
            margin=dict(l=0, r=0, b=0, t=40),
            showlegend=True
        )
        
        fig.show()


In [ ]:
# Example usage: visualize the top 3 building graphs with the most nodes
if 'dataset' in locals():
    visualize_top_k_graphs(dataset, k=3)
else:
    print("Dataset is not loaded. Please run the notebook cells above to load the dataset.")